https://github.com/mismaria/SubcellularLocationPrediction/blob/master/ProteinLocationPrediction.ipynb

To predict protein subcellular localization

In [1]:
from Bio import SeqIO
from Bio.Seq import MutableSeq
from Bio.SeqFeature import SeqFeature, FeatureLocation
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils.ProtParam import ProtParamData
from quantiprot.metrics.aaindex import get_aa2volume, get_aa2hydropathy
from quantiprot.metrics.basic import average

from Bio.SeqUtils import IsoelectricPoint
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import svm
from sklearn.preprocessing import OneHotEncoder
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
import os

Example: https://github.com/mismaria/SubcellularLocationPrediction/blob/master/ProteinLocationPrediction.ipynb
Example 2: ~/Documents/data_analysis/SSOT/Codes_SSOT/protein_subcelluar_localization_analysis/eukaryotic-proteins-master/bio

In [2]:
main_dir = "/home/joshua/Documents/data_analysis/SSOT/Codes_SSOT/protein_subcelluar_localization_analysis/Data/"

os.chdir(f'{main_dir}protein_sequence')
os.getcwd()


'/home/joshua/Documents/data_analysis/SSOT/Codes_SSOT/protein_subcelluar_localization_analysis/Data/protein_sequence'

Import training datasets
Example: https://www.biostars.org/p/9480935/

In [4]:

plasma_mem = [(protein_seq, "plasma_mem") for protein_seq in SeqIO.parse("genes_plasma_membrane.fasta", "fasta")]

cyto = [(protein_seq, "cyto") for protein_seq in SeqIO.parse("genes_cytosol.fasta", "fasta")]

mito = [(protein_seq, "mito") for protein_seq in SeqIO.parse("genes_mitochondria.fasta", "fasta")]

golgi = [(protein_seq, "golgi") for protein_seq in SeqIO.parse("genes_golgi_apparatus.fasta", "fasta")]

ER= [(protein_seq, "ER") for protein_seq in SeqIO.parse("genes_endoplasmic_reticulum.fasta", "fasta")]

vesicles= [(protein_seq, "vesicles") for protein_seq in SeqIO.parse("genes_vesicles.fasta", "fasta")]
    
nuc_bodies = [(protein_seq, "nuc_bodies") for protein_seq in SeqIO.parse("genes_nuclear_bodies.fasta", "fasta")]

nuc_fibrilar = [(protein_seq, "nuc_fibrilar") for protein_seq in SeqIO.parse("genes_nucleoli_fibrillar_center.fasta", "fasta")]

nuc_nucleoli = [(protein_seq, "nucleoli") for protein_seq in SeqIO.parse("genes_nucleoli.fasta", "fasta")]

nucleoplasma = [(protein_seq, "nucleoplasma") for protein_seq in SeqIO.parse("genes_nucleoplasma_SP.fasta", "fasta")]

nucleus = nuc_bodies + nuc_fibrilar + nuc_nucleoli + nucleoplasma

for i in range(len(nucleus)):
    lst = list(nucleus[i])
    lst[1] = "nucleus"
    nucleus[i] = tuple(lst)

secreted = [(protein_seq, "secreted") for protein_seq in SeqIO.parse("genes_secreted.fasta", "fasta")]

all_proteins = plasma_mem + cyto + mito + golgi + ER + vesicles + nucleus + secreted


print('plasma mem', len(plasma_mem))

print('cyto', len(cyto))
print('mito', len(mito))
print('golgi', len(golgi))
print('ER', len(ER))
print('vesicles', len(vesicles))
print('secreted', len(secreted))
print('nucleus', len(nucleus))



for loc in [plasma_mem, cyto, mito, golgi, ER, vesicles, nucleus, secreted]:
    print(loc, ": ", len(loc))


cyto[2][0].description.split('=')[1].replace(' GN', '')
mito[2][0].description.split('=')[1].replace(' GN', '')


cyto 1380
mito 718
secreted 1491
nucleus 3617


'Homo sapiens OX'

Extract protein features
calculate properties of amino acids

In [ ]:
# kd = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, 
#       "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5, 
#       "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6, 
#       "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}

def protein_feature_extraction(protein, option):
    protein_seq = str(MutableSeq(str(protein.seq))).replace("X", "")
    protein_seq = protein_seq.replace("U", "C")
    protein_seq = protein_seq.replace("B", "N")
    protein_seq = MutableSeq(protein_seq).toseq()

    ### features
    seq_length = len(str(protein_seq))
    analysed_seq = ProteinAnalysis(str(protein_seq))
    molecular_wt = analysed_seq.molecular_weight()
    aa_pct = analysed_seq.get_amino_acids_percent().values()
    isoelectric_points = analysed_seq.isoelectric_point()
    count = analysed_seq.count_amino_acids().values()
    aromaticity = analysed_seq.aromaticity()
    instability_index = analysed_seq.instability_index()
    hydrophobicity = analysed_seq.protein_scale(window=7, param_dict=ProtParamData.kd)
    sec_structure_fraction = analysed_seq.secondary_structure_fraction()
    epsilon_prot = analysed_seq.molar_extinction_coefficient()  # [reduced, oxidized]
    if option == "opt_1": # works core svm:  0.5065
        return np.array([seq_length, molecular_wt, isoelectric_points, instability_index] + list(sec_structure_fraction) + list(count) + list(aa_pct))
    elif option == "opt_2":
        return np.array([isoelectric_points, aromaticity,  instability_index ] + list(aa_pct) + list(count) + list(hydrophobicity) + list(sec_structure_fraction) +   list(epsilon_prot))
    elif option == "opt_3":
        return np.array([isoelectric_points, aromaticity,  instability_index ] + list(aa_pct) + list(hydrophobicity) + list(sec_structure_fraction) +   list(epsilon_prot))
    elif option == "opt_4":
        return np.array([isoelectric_points, aromaticity,  instability_index ] + list(aa_pct) + list(hydrophobicity) + list(sec_structure_fraction) )
    elif option == "opt_5": # works score svm:  0.49632
        return np.array([isoelectric_points, aromaticity,  instability_index ] + list(aa_pct) + list(sec_structure_fraction) )
    elif option == "opt_6": # works score svm:  0.5048
        return np.array([isoelectric_points, aromaticity,  instability_index ] + list(aa_pct) + list(sec_structure_fraction) + list(epsilon_prot))
    elif option == "opt_7": # works score svm:  0.32880
        return np.array([isoelectric_points] )
    elif option == "opt_8": # works score svm:  0.3684
        return np.array([ aromaticity] )
    elif option == "opt_9": # works score svm:  0.36106
        return np.array([instability_index ])
    elif option == "opt_10": # works score svm:  0.4940
        return np.array(list(aa_pct) )
    elif option == "opt_11": # works score svm:  0.4069
        return np.array(list(sec_structure_fraction) )
    elif option == "opt_12": # works score svm:  0.4006
        return np.array(list(epsilon_prot))



# for i in range(len(pro_features)):
#     print(len(pro_features[i]))


Labels the locations of the proteins

In [ ]:
for opt in ["opt_1", "opt_5", "opt_6", "opt_7", "opt_8", "opt_9", "opt_10", "opt_11", "opt_12"]:
    pro_features = ([protein_feature_extraction(pro_seq, option = opt) for pro_seq, _ in all_proteins])
    pro_loc = ([loc for _, loc in all_proteins])

    scaler = preprocessing.StandardScaler().fit(pro_features)
    pro_features = scaler.transform(pro_features)
    X_train, X_test, y_train, y_test = train_test_split(pro_features, pro_loc, test_size=0.15, random_state=0)

    ### SVM
    clf = SVC(probability=True).fit(X_train, y_train)
    print("score svm: ", clf.score(X_test, y_test), " from ", opt)

    model2=svm.NuSVC(nu=0.1, decision_function_shape='ovo', probability=True)
    model2.fit(X_train, y_train)
    print('NuSVC score:', model2.score(X_test, y_test),  " from ", opt)

    clf_rforest = RandomForestClassifier(n_estimators=10)
    clf_rforest = clf_rforest.fit(X_train, y_train)
    print("score random forest: ", clf_rforest.score(X_test, y_test), " from ", opt)





model2=svm.NuSVC(nu=0.1, decision_function_shape='ovo', probability=True)
model2.fit(X_train, y_train)
print('NuSVC score:', model2.score(X_test, y_test))

clf_rforest = RandomForestClassifier(n_estimators=10)
clf_rforest = clf_rforest.fit(X_train, y_train)
print("score random forest: ", clf_rforest.score(X_test, y_test))



pro_features = ([protein_feature_extraction(pro_seq, option = "opt_6") for pro_seq, _ in all_proteins])
pro_loc = ([loc for _, loc in all_proteins])

## cross validation
## normalize features before SVM
scaler = preprocessing.StandardScaler().fit(pro_features)
pro_features = scaler.transform(pro_features)
X_train, X_test, y_train, y_test = train_test_split(pro_features, pro_loc, test_size=0.15, random_state=0)

### SVM
clf = SVC(probability=True).fit(X_train, y_train)
print("score svm: ", clf.score(X_test, y_test))


model2=svm.NuSVC(nu=0.1, decision_function_shape='ovo', probability=True)
model2.fit(X_train, y_train)
print('NuSVC score:', model2.score(X_test, y_test))

clf_rforest = RandomForestClassifier(n_estimators=10)
clf_rforest = clf_rforest.fit(X_train, y_train)
print("score random forest: ", clf_rforest.score(X_test, y_test))




## prepare test data
test_proteins = SeqIO.parse("blind.fasta", "fasta")
testing1 = [(test.id, protein_feature_extraction(test, option= "opt_6")) for test in test_proteins]
testing = [[id, scaler.transform(test)] for id, test in testing1]
test_features = [f for _,f in testing]
test_ids = [id for id,_ in testing]

## make predictions for blind data set
blind_predictions = clf.predict_proba(test_features)

## print out results
l = ['Cyto', 'Mito', 'Nucleus', 'Secret']
for index in range(20):
    p = blind_predictions[index].tolist()
    print(test_ids[index], l[p.index(max(p))], "Confidence", int(round(max(p) * 100 )), "%")
for id, test in testing1:
    print(test)
## try out random forest
### Random Forest
##
# clf_rforest = RandomForestClassifier(n_estimators=10)
# clf_rforest = clf_rforest.fit(X_train, y_train)
# print("score random forest: ", clf_rforest.score(X_test, y_test))

In [ ]:
############################################
### labels for each dataset
labels=np.zeros(n_proteins)

# labels[0:len(cyto)]=1
# labels[len(cyto):len(cyto)+len(mito)]=2
# labels[len(cyto)+len(mito):len(cyto)+len(mito)+len(nucleus)]=3

for i in range(len(locations)):

    l = i + 1

    start = 0
    for j in range(i):
        start += len(locations[j])

    end = 0
    for j in range(l):
        end += len(locations[j])

    labels[start:end] = l

enc=OneHotEncoder(labels)

enc.categories


Model testing

In [ ]:

# Model exploration
# Comparing results for different models and for different datasets
### nu = 0.05 is the best model
model2=svm.NuSVC(nu=0.05, decision_function_shape='ovo', probability=True)
model2.fit(aa_pct, enc.categories)
print('NuSVC trained on aa_pct, score:', model2.score(aa_pct, enc.categories))


### Cross validation score
# cross validation score is very poor
cv_score = cross_val_score(model2, aa_pct, enc.categories)
print("CV average score: ", cv_score.mean())

#grid search for best parameter (nu):
indexes=np.arange(0.01,0.05,0.01)
exploreScores=np.zeros(len(indexes))
for i,nu in enumerate(indexes):
    modelx=svm.NuSVC(nu=nu, decision_function_shape='ovo', probability=True)
    modelx.fit(aa_pct, enc.categories)
    exploreScores[i]=modelx.score(aa_pct, enc.categories)


print(exploreScores)
print('bestScore:',np.max(exploreScores))
print(np.argmax(exploreScores))
best=indexes[int(np.argmax(exploreScores))]
print('best parameter:',best)  

# Use the best nu
model=svm.NuSVC(nu=np.round(best,2), decision_function_shape='ovo', probability=True)
model.fit(aa_pct, enc.categories)
print('NuSVC trained on aa_pct with best parameter, score:', model.score(aa_pct, enc.categories))


model11=svm.SVC(C=1.0,decision_function_shape='ovo', probability=True)
model11.fit(aa_pct, enc.categories)
print('SVC trained on aa_pct, score:', model11.score(aa_pct, enc.categories))

#grid search for best parameter:
indexesC=np.arange(1,30,2)
searchScores=np.zeros(len(indexesC))
for i,c in enumerate(indexesC):
    model=svm.SVC(C=c,decision_function_shape='ovo', probability=True)
    model.fit(aa_pct, enc.categories)
    searchScores[i]=model.score(aa_pct, enc.categories)

print(searchScores)
print('bestScore:',np.max(searchScores))
print(np.argmax(searchScores))
bestC=indexesC[int(np.argmax(searchScores))]
print('best parameter:',bestC)

model3=svm.SVC(C=bestC,decision_function_shape='ovo', probability=True)
model3.fit(aa_pct, enc.categories)
print('SVC trained on aa_pct, score:', model3.score(aa_pct, enc.categories))


#best models: model and model6
print('model parameters:',)
model3.get_params()









### To be deleted

raw_mcyto = SeqIO.parse("cyto.fasta", "fasta");
training_cyto = [(seq_record, "cyto") for seq_record in raw_data_cyto]

# print("\n mito");
raw_data_mito = SeqIO.parse("mito.fasta", "fasta");
training_mito = [(seq_record, "mito") for seq_record in raw_data_mito]

# print("\n nucleus");
raw_data_nucleus = SeqIO.parse("nucleus.fasta", "fasta");
training_nucleus = [(seq_record, "nucleus") for seq_record in raw_data_nucleus]

# print("\n secreted");
raw_data_secreted = SeqIO.parse("secreted.fasta", "fasta");
training_secreted = [(seq_record, "secreted") for seq_record in raw_data_secreted]

# test data
blind_tests = SeqIO.parse("blind.fasta", "fasta");

training = training_mito;
training.extend(training_cyto);
training.extend(training_nucleus);
training.extend(training_secreted);

def bio_feat(record):
    clean_seq = str(MutableSeq(str(record.seq))).replace("X", "")
    clean_seq = clean_seq.replace("U", "C")
    clean_seq = clean_seq.replace("B", "N")
    clean_seq = MutableSeq(clean_seq).toseq()

    ### features
    seq_length = len(str(clean_seq))
    analysed_seq = ProteinAnalysis(str(clean_seq))
    molecular_weight = analysed_seq.molecular_weight()
    amino_percent = analysed_seq.get_amino_acids_percent().values()
    isoelectric_points = analysed_seq.isoelectric_point()
    count = analysed_seq.count_amino_acids().values()
    # aromaticity = analysed_seq.aromaticity()  
    instability_index = analysed_seq.instability_index()
    # hydrophobicity = analysed_seq.protein_scale(ProtParamData.kd, 5, 0.4)
    secondary_structure_fraction = analysed_seq.secondary_structure_fraction()

    return np.array([seq_length, molecular_weight, isoelectric_points, instability_index] + list(secondary_structure_fraction) + list(count) + list(amino_percent))

features = ([bio_feat(record) for record, _ in training])
labels = ([label for _, label in training])

### cross validation
## normalize features before SVM
scaler = preprocessing.StandardScaler().fit(features)
features = scaler.transform(features)
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.4, random_state=0)

### SVM
clf = SVC(probability=True).fit(X_train, y_train)
print("score svm: ", clf.score(X_test, y_test))

## prepare test data
testing1 = [(test.id, bio_feat(test)) for test in blind_tests]
testing = [[id, scaler.transform(test)] for id, test in testing1]
test_features = [f for _,f in testing]
test_ids = [id for id,_ in testing]

## make predictions for blind data set
blind_predictions = clf.predict_proba(test_features)

## print out results
l = ['Cyto', 'Mito', 'Nucleus', 'Secret']
for index in range(20):
    p = blind_predictions[index].tolist()
    print(test_ids[index], l[p.index(max(p))], "Confidence", int(round(max(p) * 100 )), "%")

## try out random forest
### Random Forest
##
# clf_rforest = RandomForestClassifier(n_estimators=10)
# clf_rforest = clf_rforest.fit(X_train, y_train)
# print("score random forest: ", clf_rforest.score(X_test, y_test))

Model trained on protein sequence

In [ ]:
model8=svm.SVC(decision_function_shape='ovo', probability=True)
model8.fit(protein_length, enc.categories)
print('SVC trained on sequence length, score:', model8.score(protein_length, enc.categories))

NuSVC model trained on protein sequecne

In [ ]:
model10=svm.NuSVC(nu=0.5, decision_function_shape='ovo', probability=True)
model10.fit(protein_length,, enc.categories)
print('NuSVC trained on sequence length, score:', model10.score(protein_length, enc.categories))

Model trained on isoelectric point

In [ ]:
model9=svm.SVC(decision_function_shape='ovo', probability=True)
model9.fit(iep, enc.categories)
print('SVC trained on isoelectric point, score:', model9.score(iep, enc.categories))

NuSVC model trained on isoelectric point

In [ ]:
model11=svm.NuSVC(nu=0.05, decision_function_shape='ovo', probability=True)
model11.fit(ie_point, enc.categories)
print('NuSVC trained on isoelectric point, score:', model11.score(ie_point, enc.categories))

Model trained on all feactures

In [ ]:
model4=svm.SVC(C=10.0,decision_function_shape='ovo', probability=True)
model4.fit(select_features, enc.categories)
print('SVC trained on all features, score:', model4.score(select_features, enc.categories))

model44=svm.NuSVC(nu=0.05, decision_function_shape='ovo', probability=True)
model44.fit(select_features, enc.categories)
print('NuSVC trained on select features, score:', model44.score(select_features, enc.categories))

model2=svm.NuSVC(nu=0.05, decision_function_shape='ovo', probability=True)
model2.fit(aa_pct, enc.categories)
print('NuSVC trained on aa_pct, score:', model2.score(aa_pct, enc.categories))

One classifier

In [ ]:
#one classifier per class
model6 = OneVsRestClassifier(model).fit(aa_pct, enc.categories)
print('OneVsRestClassifier trained on all model1, score:', model6.score(aa_pct, enc.categories))

#best models: model and model6
print('model parameters:',)
model.get_params()

print('model6 parameters:',)
model6.get_params()

Cross validation

In [ ]:
predsTrain1=model.predict(aa_pct)
corrTrain1=np.corrcoef(predsTrain1, enc.categories)
probsTrain1=model.predict_proba(aa_pct)
predProbsTrain1=np.array(np.argmax(probsTrain1, axis=1)+1)
conf1=np.array(np.max(probsTrain1, axis=1))
print('score 1:', model.score(aa_pct, enc.categories))
print('correlation 1:',corrTrain1[1,0])
print('average confidence 1:', np.average(conf1))

### The best model 
predsTrain2=model2.predict(aa_pct)
corrTrain2=np.corrcoef(predsTrain2, enc.categories)
probsTrain2=model2.predict_proba(aa_pct)
predProbsTrain2=np.array(np.argmax(probsTrain2, axis=1)+1)
conf2=np.array(np.max(probsTrain2, axis=1))
print('score 2:', model2.score(aa_pct, enc.categories))
print('correlation 2:',corrTrain2[1,0])
print('average confidence 2:', np.average(conf2))


predsTrain6=model6.predict(aa_pct)
corrTrain6=np.corrcoef(predsTrain6, enc.categories)
probsTrain6=model6.predict_proba(aa_pct)
print('score 6:', model6.score(aa_pct, enc.categories))
print('correlation 6:',corrTrain6[1,0])
predProbsTrain6=np.array(np.argmax(probsTrain6, axis=1)+1)
conf6=np.array(np.max(probsTrain6, axis=1))
print('average confidence 6:', np.average(conf6))


In [ ]:
## if the two models agree then great, otherwise pick the prediction with max probability

print("Compare two predictions:")
myPredsTrain=np.zeros((len(protein_analysis_res), 2))

for i in range(len(protein_analysis_res)):
    if predsTrain1[i]==predsTrain6[i]:
        ans=predsTrain1[i]
        ansprob=max(probsTrain1[i, int(predsTrain6[i]-1)], probsTrain6[i, int(predsTrain6[i]-1)])
        print('index:', i, 'prediction:', ans, 'with prob', ansprob)
    else:
        a=np.max(probsTrain1[i])
        b=np.max(probsTrain6[i])
        if a>b:
            ans=np.argmax(probsTrain1[i])+1
            ansprob=np.max(probsTrain1[i])
        else:
            ans=np.argmax(probsTrain6[i])+1
            ansprob=np.max(probsTrain6[i])
        print('index:',i, 'decide', ans, 'with prob', ansprob)
    myPredsTrain[i]=ans,ansprob

In [ ]:
totalCorrect=0
for i in range(len(protein_analysis_res)):    
    if myPredsTrain[i][0]==enc.categories[i]:
        totalCorrect+=1
print('correct assignments:', totalCorrect)
print('score:', totalCorrect/len(protein_analysis_res)*100)
finalCorr=np.corrcoef(myPredsTrain[:,0], enc.categories)
print('correlation coefficient:', finalCorr[1,0])
avgConf=np.average(myPredsTrain[:,1])
print('average confidence:', avgConf)

In [ ]:
def KF_svc(X,y,cost=1.0,gamma='scale',k=10):
    kf = KFold(n_splits=k, shuffle=True)
    model_scores=np.zeros(k)
    corr=np.zeros(k)
    #iterator
    it=iter(np.arange(k))
    
    model=svm.SVC(C=cost, kernel='rbf', gamma=gamma, probability=True)
    for train_index, test_index in kf.split(X, y):
        #split the dataset into two parts: (k-1)/k*n and (1/k)*n
        i=next(it)
        print('iteration', i)
        
        #split data
        X_train, X_test=X[train_index], X[test_index]
        y_train, y_test=y[train_index], y[test_index]
        #fit model
        model.fit(X_train, y_train)
        model_scores[i]=model.score(X_test,y_test)
        pred=model.predict(X_test)
        corr[i]=np.corrcoef(pred, y_test)[1,0]
        
    return model, model_scores, corr

In [ ]:
model7, score7, corr7=KF_svc(aa_pct, enc.categories);

In [ ]:
for i in range(10):
    print(i, 'score:', np.round(score7[i],3), 'correlation:', np.round(corr7[i],3))
print('average score:', np.mean(score7))
print('average coefficient:', np.mean(corr7))

In [ ]:
labels=range(1,5)
rates=confusion_matrix(y_true=enc.categories, y_pred= myPredsTrain[:,0], labels=labels)
#plot heatmap
cmap = 'hot'
cmap='Blues'

plt.figure(figsize=(8, 6))
tick_marks = np.arange(len(labels))
plt.xticks(tick_marks, labels)
plt.yticks(tick_marks, labels)
plt.imshow(rates, cmap, interpolation='nearest')
plt.title('Confusion Matrix')
plt.colorbar()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

In [ ]:
predictions=myPredsTrain[:,0]
x=np.arange(len(dataCyto))
y1=predictions[0:len(dataCyto)]
y2=enc.categories[0:len(dataCyto)]
x1=x
x2=x
plt.plot(x1, y1, 'ro',label = "predictions")
plt.plot(x2, y2, label = "labels")
plt.xlabel('protein')
plt.ylabel('categories')
plt.title('compare predictions with labels\nCategory 1-Cytosolic')
plt.legend()
plt.figure(figsize=(30,10))
plt.show()

x=np.arange(len(dataCyto),len(dataCyto)+len(dataMito))
y1=predictions[len(dataCyto):len(dataCyto)+len(dataMito)]
y2=enc.categories[len(dataCyto):len(dataCyto)+len(dataMito)]
x1=x
x2=x
plt.plot(x1, y1, 'ro',label = "predictions")
plt.plot(x2, y2, label = "labels")
plt.xlabel('protein')
plt.ylabel('categories')
plt.title('compare predictions with labels\nCategory- Mitochondrial')
plt.legend()
plt.figure(figsize=(30,10))
plt.show()

x=np.arange(len(dataCyto)+len(dataMito),len(dataCyto)+len(dataMito)+len(dataNucleus))
y1=predictions[len(dataCyto)+len(dataMito):len(dataCyto)+len(dataMito)+len(dataNucleus)]
y2=enc.categories[len(dataCyto)+len(dataMito):len(dataCyto)+len(dataMito)+len(dataNucleus)]
x1=x
x2=x
plt.plot(x1, y1, 'ro',label = "predictions")
plt.plot(x2, y2, label = "labels")
plt.xlabel('protein')
plt.ylabel('categories')
plt.title('compare predictions with labels\nCategory 3-Nuclear')
plt.legend()
plt.figure(figsize=(30,10))
plt.show()

x=np.arange(len(dataCyto)+len(dataMito)+len(dataNucleus), 9222)
y1=predictions[len(dataCyto)+len(dataMito)+len(dataNucleus):]
y2=enc.categories[len(dataCyto)+len(dataMito)+len(dataNucleus):]
x1=x
x2=x
plt.plot(x1, y1, 'ro',label = "predictions")
plt.plot(x2, y2, label = "labels")
plt.xlabel('protein')
plt.ylabel('categories')
plt.title('compare predictions with labels\nCategory 4-Secreted')
plt.legend()
plt.figure(figsize=(30,10))
plt.show()




Testing
Make some predictions

In [ ]:
print('model 1:')
print('prediction:', model.predict([aa_pct[0]]))
print('probClass1, probClass2, probClass3, probClass 4')
print(model.predict_proba([aa_pct[0]]))
print('\nmodel 6:')
print('prediction:', model6.predict([aa_pct[0]]))
print('probClass1, probClass2, probClass3, probClass 4')
print(model6.predict_proba([aa_pct[0]]))

Analyse test data

In [ ]:
dataBlind=[]
for record in SeqIO.parse("blind.fasta", "fasta"):
# for record in SeqIO.parse("blind.fasta.txt", "fasta"):
    dataBlind.append(record)
print('blind:', len(dataBlind))


blind=[]
for i in range(len(dataBlind)):
    analysed_seq=ProteinAnalysis(str(dataBlind[i].seq))
    blind.append(analysed_seq)


compBlind=np.zeros((len(dataBlind), 20))
for i in range(len(dataBlind)):
    dict=blind[i].get_amino_acids_percent()
    for pos,aa in enumerate(dict.values()):
        compBlind[i,pos]=aa



Calculcate predictions for NuSVC model

In [ ]:
preds=np.zeros(len(dataBlind))
for i in range(len(dataBlind)):
    preds[i]=model.predict([compBlind[i]])
    
probs=np.zeros((len(dataBlind), 4))
for i in range(len(dataBlind)):
    probs[i]=model.predict_proba([compBlind[i]])
    
predProbs=np.array(np.argmax(probs, axis=1)+1)

predProbs



print("Predictions for SVC model")
for i in range(len(dataBlind)):
    if preds[i]==predProbs[i]:
        print('index:', i, 'prediction:', preds[i])
    else:
        print('index:',i, 'predicts:', preds[i], 'from probability:', predProbs[i], probs[i, int(predProbs[i]-1)])


for i in range(len(dataBlind)):
    if preds[i]==1:
        print(dataBlind[i].name, 'Cytosolic, Confidence', np.round(np.max(probs[i])*100), '%')
    elif preds[i]==2:
        print(dataBlind[i].name, 'Mitochondrial, Confidence', np.round(np.max(probs[i])*100), '%')
    elif preds[i]==3:
        print(dataBlind[i].name, 'Nuclear, Confidence', np.round(np.max(probs[i])*100), '%')
    elif preds[i]==4:
        print(dataBlind[i].name, 'Secreted, Confidence', np.round(np.max(probs[i])*100), '%')



Calculcate predictions for combined model

In [ ]:
preds6=np.zeros(len(dataBlind))
for i in range(len(dataBlind)):
    preds6[i]=model6.predict([compBlind[i]])
    
probs6=np.zeros((len(dataBlind), 4))
for i in range(len(dataBlind)):
    probs6[i]=model6.predict_proba([compBlind[i]])
    
predProbs6=np.array(np.argmax(probs6, axis=1)+1)


print("Predictions for combined model")
for i in range(len(dataBlind)):
    if preds6[i]==predProbs6[i]:
        print('index:', i, 'prediction:', preds6[i])
    else:
        print('index:',i, 'predicts:', preds6[i], 'from probability:', predProbs6[i], probs6[i,int(predProbs6[i]-1)])



for i in range(len(dataBlind)):
    if preds6[i]==1:
        print(dataBlind[i].name, 'Cytosolic, Confidence', np.round(np.max(probs6[i])*100), '%')
    elif preds6[i]==2:
        print(dataBlind[i].name, 'Mitochondrial, Confidence', np.round(np.max(probs6[i])*100), '%')
    elif preds6[i]==3:
        print(dataBlind[i].name, 'Nuclear, Confidence', np.round(np.max(probs6[i])*100), '%')
    elif preds6[i]==4:
        print(dataBlind[i].name, 'Secreted, Confidence', np.round(np.max(probs6[i])*100), '%')




Compare the two predictions

In [ ]:
## if the two models agree then great, otherwise pick the prediction with max probability
print("Compare two predictions:")
myPreds=np.zeros((len(dataBlind), 2))

for i in range(len(dataBlind)):
    if preds[i]==preds6[i]:
        ans=preds[i]
        ansprob=max(probs[i, int(preds[i]-1)], probs6[i, int(preds6[i]-1)])
        print('index:', i, 'prediction:', ans, 'with prob', ansprob)
    else:
        a=np.max(probs[i])
        b=np.max(probs6[i])
        if a>b:
            ans=np.argmax(probs[i])+1
            ansprob=np.max(probs[i])
        else:
            ans=np.argmax(probs6[i])+1
            ansprob=np.max(probs6[i])
        print('index:',i, 'decide', ans, 'with prob', ansprob)
    myPreds[i]=ans,ansprob

print(myPreds)

Final predictions

In [ ]:
for i in range(len(dataBlind)):
    if myPreds[i,0]==1:
        print(dataBlind[i].name, 'Cytosolic, Confidence', np.round(myPreds[i,1]*100, 1), '%')
    elif myPreds[i,0]==2:
        print(dataBlind[i].name, 'Mitochondrial, Confidence', np.round(myPreds[i,1]*100,1), '%')
    elif myPreds[i,0]==3:
        print(dataBlind[i].name, 'Nuclear, Confidence', np.round(myPreds[i,1]*100,1), '%')
    elif myPreds[i,0]==4:
        print(dataBlind[i].name, 'Secreted, Confidence', np.round(myPreds[i,1]*100,1), '%')




Further testing

Analyse the data

In [ ]:
furtherData=[]

####change the filename to match the file you want to test
##For further information see README.md
for record in SeqIO.parse("filename.txt", "fasta"):
    furtherData.append(record)
print('furtherData:', len(furtherData))

furtherSequences=[]
for i in range(len(furtherData)):
    analysed_seq=ProteinAnalysis(str(furtherData[i].seq))
    furtherSequences.append(analysed_seq)
    
compFurther=np.zeros((len(furtherData), 20))
for i in range(len(furtherData)):
    dict=furtherSequences[i].get_amino_acids_percent()
    for pos,aa in enumerate(dict.values()):
        compFurther[i,pos]=aa



Make predictions on further data

In [ ]:
#model 1
predsF=np.zeros(len(furtherData))
for i in range(len(furtherData)):
    predsF[i]=model.predict([compFurther[i]])
    
probsF=np.zeros((len(furtherData), 4))
for i in range(len(furtherData)):
    probsF[i]=model.predict_proba([compFurther[i]])
    
predProbsF=np.array(np.argmax(probsF, axis=1)+1)

#model 6
preds6F=np.zeros(len(furtherData))
for i in range(len(furtherData)):
    preds6F[i]=model6.predict([compFurther[i]])
    
probs6F=np.zeros((len(furtherData), 4))
for i in range(len(furtherData)):
    probs6F[i]=model6.predict_proba([compFurther[i]])
    
predProbs6F=np.array(np.argmax(probs6F, axis=1)+1)


print("Predictions:")
myPredsF=np.zeros((len(furtherData), 2))

for i in range(len(furtherData)):
    if predsF[i]==preds6F[i]:
        ans=predsF[i]
        ansprob=max(probsF[i, int(predsF[i]-1)], probs6F[i, int(preds6F[i]-1)])
    else:
        a=np.max(probsF[i])
        b=np.max(probs6F[i])
        if a>b:
            ans=np.argmax(probsF[i])+1
            ansprob=np.max(probsF[i])
        else:
            ans=np.argmax(probs6F[i])+1
            ansprob=np.max(probs6F[i])
    myPredsF[i]=ans,ansprob
    
for i in range(len(furtherData)):
    if myPredsF[i,0]==1:
        print(furtherData[i].name, 'Cytosolic, Confidence', np.round(myPredsF[i,1]*100), '%')
    elif myPredsF[i,0]==2:
        print(furtherData[i].name, 'Mitochondrial, Confidence', np.round(myPredsF[i,1]*100), '%')
    elif myPredsF[i,0]==3:
        print(furtherData[i].name, 'Nuclear, Confidence', np.round(myPredsF[i,1]*100), '%')
    elif myPredsF[i,0]==4:
        print(furtherData[i].name, 'Secreted, Confidence', np.round(myPredsF[i,1]*100), '%')